<a href="https://colab.research.google.com/github/AbhinavS0201/ArchaeoMind/blob/main/ArchaeoMind_CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install --upgrade kagglehub

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("abhinavrama22/archaeomind-images")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'archaeomind-images' dataset.
Path to dataset files: /kaggle/input/archaeomind-images


In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.models import Model
import os

In [ ]:
import os

BASE_DIR = "/kaggle/input/archaeomind-images/ArchaeoMind_dataset"

print(os.listdir(BASE_DIR))


['val', 'test', 'train']


In [ ]:

TRAIN_DIR = os.path.join(BASE_DIR, "train")
VAL_DIR   = os.path.join(BASE_DIR, "val")
TEST_DIR  = os.path.join(BASE_DIR, "test")


In [ ]:
train_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True
)

val_test_gen = ImageDataGenerator(rescale=1./255)


In [ ]:

train_data = train_gen.flow_from_directory(
    TRAIN_DIR,
    target_size=(224, 224),
    batch_size=32,
    class_mode="binary"
)


val_data = val_test_gen.flow_from_directory(
    VAL_DIR,
    target_size=(224, 224),
    batch_size=16,
    class_mode='binary'
)

test_data = val_test_gen.flow_from_directory(
    TEST_DIR,
    target_size=(224, 224),
    batch_size=16,
    class_mode='binary',
    shuffle=False
)


Found 849 images belonging to 2 classes.
Found 202 images belonging to 2 classes.
Found 201 images belonging to 2 classes.


In [ ]:
base_model = MobileNetV2(
    input_shape=(224,224,3),
    include_top=False,
    weights="imagenet"
)

base_model.trainable = False


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
print(train_data.class_indices)

{'artifact': 0, 'non_artifact': 1}


In [ ]:
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.3)(x)
output = Dense(1, activation='sigmoid')(x)

model = Model(inputs=base_model.input, outputs=output)


In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 112, 112,  │        864 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 112, 112,  │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 112, 112,  │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 112, 112,  │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 112, 112,  │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 112, 112,  │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 113, 113,  │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 56, 56,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │          0 │ block_1_depthwis… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 56, 56,    │      2,304 │ block_1_depthwis

 Total params: 2,422,081 (9.24 MB)

 Trainable params: 164,097 (641.00 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [ ]:
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=10
)

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10
27/27 ━━━━━━━━━━━━━━━━━━━━ 62s 2s/step - accuracy: 0.6378 - loss: 0.6724 - val_accuracy: 0.6881 - val_loss: 0.6095
Epoch 2/10
27/27 ━━━━━━━━━━━━━━━━━━━━ 52s 2s/step - accuracy: 0.8330 - loss: 0.4209 - val_accuracy: 0.7921 - val_loss: 0.5416
Epoch 3/10
27/27 ━━━━━━━━━━━━━━━━━━━━ 53s 2s/step - accuracy: 0.8757 - loss: 0.3192 - val_accuracy: 0.8267 - val_loss: 0.5158
Epoch 4/10
27/27 ━━━━━━━━━━━━━━━━━━━━ 53s 2s/step - accuracy: 0.8881 - loss: 0.2670 - val_accuracy: 0.8168 - val_loss: 0.5065
Epoch 5/10
27/27 ━━━━━━━━━━━━━━━━━━━━ 53s 2s/step - accuracy: 0.9097 - loss: 0.2552 - val_accuracy: 0.8267 - val_loss: 0.5026
Epoch 6/10
27/27 ━━━━━━━━━━━━━━━━━━━━ 55s 2s/step - accuracy: 0.9109 - loss: 0.2219 - val_accuracy: 0.8267 - val_loss: 0.4949
Epoch 7/10
27/27 ━━━━━━━━━━━━━━━━━━━━ 65s 2s/step - accuracy: 0.9421 - loss: 0.2053 - val_accuracy: 0.8267 - val_loss: 0.4911
Epoch 8/10
27/27 ━━━━━━━━━━━━━━━━━━━━ 53s 2s/step - accuracy: 0.9379 - loss: 0.1721 - val_accuracy: 0.8267 - val_loss:

In [ ]:
train_acc = history.history['accuracy'][-1]
val_acc = history.history['val_accuracy'][-1]

print("Training Accuracy:", train_acc * 100, "%")
print("Validation Accuracy:", val_acc * 100, "%")


Training Accuracy: 94.93522047996521 %
Validation Accuracy: 81.18811845779419 %


In [ ]:
test_loss, test_accuracy = model.evaluate(test_data)
print("Test Accuracy:", test_accuracy)


13/13 ━━━━━━━━━━━━━━━━━━━━ 9s 673ms/step - accuracy: 0.9336 - loss: 0.2482
Test Accuracy: 0.8855721354484558


In [ ]:
model.save("archaeomind_cnn_model.h5")
print("Model saved successfully")

Model saved successfully


In [ ]:
import numpy as np
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.models import load_model

# Load trained model
model = load_model("archaeomind_cnn_model.h5")

def predict_image(image_path):
    img = load_img(image_path, target_size=(224, 224))
    img_array = img_to_array(img)
    img_array = img_array / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    prediction = model.predict(img_array)[0][0]

    #Because non_artifact = 1

    if prediction <= 0.7:
        print("Artifact Detected")
    else:
        print("Non-Artifact Detected")
    print(f"Confidence Score : {prediction:.2f}")

## Sample Inference Results
The following predictions demonstrate the model's performance on unseen sample images
(Pottery and Pebbles) to validate real-world inference behavior.


In [ ]:
predict_image("/content/Pebbles.jpeg")
predict_image("/content/Pottery.jpeg")

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
Non-Artifact Detected
Confidence Score : 1.00
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step
Artifact Detected
Confidence Score : 0.06


## Reproducibility
The dataset used in this project is publicly available on Kaggle, and the
complete implementation is available on GitHub. Minor framework warnings
may appear during execution and can be safely ignored.


In [1]:
import numpy as np
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_auc_score
)

test_data.reset()

# Get prediction probabilities
y_prob = model.predict(test_data).ravel()

# Actual labels
y_true = test_data.classes

# Standard threshold = 0.5
y_pred = (y_prob >= 0.5).astype(int)

print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred))

print("\nClassification Report:")
print(classification_report(
    y_true,
    y_pred,
    target_names=["artifact", "non_artifact"]
))

print("\nROC-AUC:")
print(roc_auc_score(y_true, y_prob))

NameError: name 'test_data' is not defined